# Build Local Knowledge Index (Experimental Phase 1)

This notebook builds a **local-only** knowledge index from files in `agent/ui/knowledge/`.

What this does:
- Loads `.txt`, `.md`, and `.pdf` files
- Splits documents into overlapping chunks
- Builds a BM25 keyword index
- Saves index artifacts in `agent/ui/storage/`

## Privacy and Design

- Indexing happens locally.
- Retrieval happens locally.
- OpenAI is **not** used for indexing or retrieval.
- Documents are **not** uploaded to OpenAI.

If optional LLM fallback is used later, only selected retrieved chunks may be included in prompts.

In [ ]:
from pathlib import Path
import sys

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    for candidate in candidates:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'agent').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate project root containing AGENTS.md and agent/.')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

KNOWLEDGE_DIR = PROJECT_ROOT / 'agent' / 'ui' / 'knowledge'
STORAGE_DIR = PROJECT_ROOT / 'agent' / 'ui' / 'storage'

KNOWLEDGE_DIR.mkdir(parents=True, exist_ok=True)
STORAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Knowledge dir: {KNOWLEDGE_DIR}')
print(f'Storage dir: {STORAGE_DIR}')

## Add Documents

Place your `.txt`, `.md`, and `.pdf` files into `agent/ui/knowledge/`, then run the next scan cell.

In [ ]:
from agent.ui.rag.loaders import scan_knowledge_files

files = scan_knowledge_files(KNOWLEDGE_DIR)
print(f'Found {len(files)} supported file(s):')
for path in files:
    print(' -', path.relative_to(PROJECT_ROOT))

if not files:
    print('No files found yet. Add documents to agent/ui/knowledge/ and rerun this cell.')

## Build the Local Index

This calls the reusable `build_local_index()` function from `agent/ui/rag/build_index.py`.

In [ ]:
from agent.ui.rag.build_index import build_local_index

summary = build_local_index(
    knowledge_dir=KNOWLEDGE_DIR,
    storage_dir=STORAGE_DIR,
    chunk_size=350,
    chunk_overlap=75,
)

print('Index build complete.')
print(f"Files indexed: {summary['num_files_indexed']}")
print(f"Chunks created: {summary['num_chunks_created']}")
print(f"chunks.jsonl: {summary['chunks_path']}")
print(f"bm25.pkl: {summary['bm25_path']}")
print(f"manifest.json: {summary['manifest_path']}")

## Test Retrieval

Enter a test question and inspect top retrieved chunks.

Tip: if you add or edit files in `agent/ui/knowledge/`, rerun the index build cell before testing retrieval again.

In [ ]:
from agent.ui.rag.retrieve import retrieve, build_context

query = 'Why did I get this score?'
k = 5

results = retrieve(query=query, k=k, storage_dir=STORAGE_DIR)
print(f'Query: {query}')
print(f'Retrieved: {len(results)} chunk(s)')

for i, item in enumerate(results, start=1):
    preview = item['text'][:220].replace('\n', ' ')
    print(f"\n[{i}] {item['source_name']} | chunk {item['chunk_index']} | score={item['score']:.4f}")
    print(preview + ('...' if len(item['text']) > 220 else ''))

context_block = build_context(results)
print('\nFormatted context preview:')
print(context_block[:1200] + ('...' if len(context_block) > 1200 else ''))

In [ ]:
from agent.ui.rag.build_index import build_local_index
from agent.ui.rag.retrieve import retrieve

SETTINGS = [
    (350, 75),
    (500, 100),
    (700, 120),
]

EVAL_QUERIES = [
    'Why did I get this score?',
    'What does CV versus test gap mean?',
    'How can I improve descriptor relevance?',
]

for chunk_size, chunk_overlap in SETTINGS:
    print('=' * 80)
    print(f'Setting: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}')

    build_local_index(
        knowledge_dir=KNOWLEDGE_DIR,
        storage_dir=STORAGE_DIR,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    for query in EVAL_QUERIES:
        results = retrieve(query=query, k=2, storage_dir=STORAGE_DIR)
        print(f"\nQuery: {query}")
        for i, item in enumerate(results, start=1):
            preview = item['text'][:180].replace('\n', ' ')
            print(
                f"  [{i}] {item['source_name']} | chunk {item['chunk_index']} | score={item['score']:.4f}"
            )
            print(f"      {preview}{'...' if len(item['text']) > 180 else ''}")

print('\nDone. Pick the setting with the most focused, relevant chunk previews for your questions.')

## How to Interpret Evaluation Output

You just compared multiple chunk settings with your own questions.

Use this rule of thumb:
1. Prefer settings where top chunks are focused and directly answer the question.
2. If top chunks are too broad, use smaller chunks.
3. If answers miss needed context, try slightly larger chunks.

Recommended starting default for this project: `chunk_size=350`, `chunk_overlap=75`.

## Next Step

Your local index is now built and retrievable.

You can now run the ROBERT UI app as usual.

Important: if you add, remove, or edit files in `agent/ui/knowledge/`, rerun the index build cell before using retrieval/chat again.

Chat integration with this local retrieval layer will be added in a later phase.

Future direction: you can also place FAQ-style local notes (questions and answers) in `agent/ui/knowledge/` to make them retrievable once chat integration is enabled.